# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from torch.utils.data import Dataset
from torch.optim import AdamW
from transformers import DataCollatorForLanguageModeling
import matplotlib as plt
from transformers import BertTokenizer, BertModel
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import re
from sklearn.model_selection import GridSearchCV

# Datasets preparation

In [7]:
df_ref = pd.read_csv("files/df_ml_good_with_features.csv")

df_ref_amy = df_ref[df_ref['class'] == 1]
df_ref_nonamy = df_ref[df_ref['class'] == 0]

df_protgpt2 = pd.read_csv("./files/final_datasets/protgpt_final.csv")
df_cvae = pd.read_csv("./files/final_datasets/cvae_final.csv")

In [8]:
selected_features = ['beta_propensity', 'proline_fraction', 'AAT', 'net_charge', 'TA', 'polar_fraction', 'a3vSA']

## Train/Test spilt

In [9]:
X_real = df_ref[selected_features]
y_real = df_ref["class"]

X_train_ref, X_test, y_train_ref, y_test = train_test_split(
    X_real, y_real,
    test_size=0.2,
    random_state=42,
    stratify=y_real
)

In [10]:
df_protgpt2['class'] = 1
df_cvae['class'] = 1

train_datasets = {
    "Real Only": (X_train_ref, y_train_ref),
    "Real + ProtGPT2": (
        pd.concat([X_train_ref, df_protgpt2[selected_features]]),
        pd.concat([y_train_ref, df_protgpt2["class"]])
    ),
    "Real + cVAE": (
        pd.concat([X_train_ref, df_cvae[selected_features]]),
        pd.concat([y_train_ref, df_cvae["class"]])
    )
}

# Models

## Baseline

In [11]:
models = {
    "RandomForest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
}

In [12]:
param_grids = {
    "RandomForest": {
        'n_estimators': [100, 300, 500],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    "XGBoost": {
        'n_estimators': [100, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 6],
        'subsample': [0.8, 1.0]
    },
    "LogisticRegression": {
        'C': [0.01, 0.1, 1.0, 10.0],
        'solver': ['lbfgs', 'liblinear'], # liblinear powinien byc dobry dla malych zbiorów
        'penalty': ['l2']
    }
}

In [13]:
results = []

for data_name, (X_train_curr, y_train_curr) in train_datasets.items():
    print(f"\n>>> Optymalizacja dla zbioru: {data_name}")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_curr)
    X_test_scaled = scaler.transform(X_test)

    for model_name, model in models.items():
        print(f"Szukanie parametrów dla {model_name}...")

        # GridSearchCV automatycznie robi Cross-Validation
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grids[model_name],
            scoring='f1', # Optymalizujemy pod F1-score
            cv=3,         # 3-krotna walidacja krzyżowa
            n_jobs=-1
        )

        grid_search.fit(X_train_scaled, y_train_curr)

        # Najlepszy model po optymalizacji
        best_model = grid_search.best_estimator_

        # Testowanie na stałym zbiorze
        probs = best_model.predict_proba(X_test_scaled)[:, 1]
        preds = best_model.predict(X_test_scaled)

        results.append({
            "Model": model_name,
            "Dataset": data_name,
            "Best_Params": grid_search.best_params_,
            "AUC": roc_auc_score(y_test, probs),
            "F1": f1_score(y_test, preds)
        })


>>> Optymalizacja dla zbioru: Real Only
Szukanie parametrów dla RandomForest...
Szukanie parametrów dla XGBoost...
Szukanie parametrów dla LogisticRegression...

>>> Optymalizacja dla zbioru: Real + ProtGPT2
Szukanie parametrów dla RandomForest...
Szukanie parametrów dla XGBoost...
Szukanie parametrów dla LogisticRegression...

>>> Optymalizacja dla zbioru: Real + cVAE
Szukanie parametrów dla RandomForest...
Szukanie parametrów dla XGBoost...
Szukanie parametrów dla LogisticRegression...


### Results - baseline

In [15]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["AUC", "F1"],
    ascending=False
)

results_df

,Model,Dataset,Best_Params,AUC,F1
0,RandomForest,Real Only,"{'max_depth': 10, 'min_samples_split': 5, 'n_e...",0.791526,0.751196
1,XGBoost,Real Only,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.783542,0.749415
2,LogisticRegression,Real Only,"{'C': 0.01, 'penalty': 'l2', 'solver': 'liblin...",0.747392,0.705882
3,RandomForest,Real + ProtGPT2,"{'max_depth': 10, 'min_samples_split': 10, 'n_...",0.741146,0.755467
4,XGBoost,Real + ProtGPT2,"{'learning_rate': 0.01, 'max_depth': 4, 'n_est...",0.738365,0.744639
5,LogisticRegression,Real + ProtGPT2,"{'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}",0.727706,0.649616
6,RandomForest,Real + cVAE,"{'max_depth': 10, 'min_samples_split': 10, 'n_...",0.712432,0.747899
7,XGBoost,Real + cVAE,"{'learning_rate': 0.01, 'max_depth': 4, 'n_est...",0.688333,0.730627
8,LogisticRegression,Real + cVAE,"{'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}",0.650778,0.477876


In [16]:
summary = results_df.pivot_table(
    index="Model",
    columns="Dataset",
    values=["AUC", "F1"]
)

summary = summary.round(3)

print(summary)

                               AUC                                    F1  \
Dataset            Real + ProtGPT2 Real + cVAE Real Only Real + ProtGPT2   
Model                                                                      
LogisticRegression           0.728       0.651     0.747           0.650   
RandomForest                 0.741       0.712     0.792           0.755   
XGBoost                      0.738       0.688     0.784           0.745   

                                          
Dataset            Real + cVAE Real Only  
Model                                     
LogisticRegression       0.478     0.706  
RandomForest             0.748     0.751  
XGBoost                  0.731     0.749  


## Deep models

MLP

In [18]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4), # Regularyzacja
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1) # Brak Sigmoida - BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x)

CNNHybrid

In [19]:
class CNNHybrid(nn.Module):
    def __init__(self, vocab_size, feat_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32, padding_idx=0)
        self.conv = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )
        self.feat_net = nn.Sequential(
            nn.Linear(feat_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )

    def forward(self, seq, feats):
        x = self.embedding(seq).permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)
        f = self.feat_net(feats)
        combined = torch.cat([x, f], dim=1)
        return self.classifier(combined)

ProtBERT MLP

In [20]:
class ProtBERT_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, _=None): # _ dla kompatybilności z pętlą (Hybrid wejście)
        return self.net(x)

In [ ]:
def preprocess_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[UZOB]", "X", seq)  # rzadkie aminokwasy
    return " ".join(list(seq))

In [ ]:
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model_bert = BertModel.from_pretrained("Rostlab/prot_bert")

def get_embeddings(sequences, batch_size=16):
    embeddings = []

    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i+batch_size]
        batch_seqs = [preprocess_sequence(seq) for seq in batch_seqs]

        encoded = tokenizer(batch_seqs,
                            return_tensors="pt",
                            padding=True,
                            truncation=True,
                            max_length=512)

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model_bert(input_ids=input_ids,
                                 attention_mask=attention_mask)

        # (batch, seq_len, hidden_dim)
        hidden_states = outputs.last_hidden_state

        # mean pooling (ignorujemy padding)
        mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        summed = torch.sum(hidden_states * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts

        embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(embeddings)

Training

In [ ]:
def run_deep_experiment(model, train_loader, val_loader, device, lr=0.001, patience=12):
    model.to(device)
    # BCEWithLogitsLoss jest stabilniejszy numerycznie
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) # L2 Regularization

    best_loss = float('inf')
    best_state = None
    counter = 0

    for epoch in range(200):
        # Faza Treningu
        model.train()
        train_loss = 0
        for seq, feat, labels in train_loader:
            seq, feat, labels = seq.to(device), feat.to(device), labels.to(device).unsqueeze(1)

            optimizer.zero_grad()
            logits = model(seq, feat) if not isinstance(model, MLP) and not isinstance(model, ProtBERT_MLP) else model(feat)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Faza Walidacji
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for seq, feat, labels in val_loader:
                seq, feat, labels = seq.to(device), feat.to(device), labels.to(device).unsqueeze(1)
                logits = model(seq, feat) if not isinstance(model, MLP) and not isinstance(model, ProtBERT_MLP) else model(feat)
                val_loss += criterion(logits, labels).item()

        avg_val_loss = val_loss / len(val_loader)

        # Early Stopping check
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    model.load_state_dict(best_state)
    return model

In [ ]:
class AmyloidDataset(Dataset):
    def __init__(self, sequences, features, labels, aa_to_idx, max_len=50):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values if hasattr(labels, 'values') else labels, dtype=torch.float32)
        self.sequences = [self._encode(s, aa_to_idx, max_len) for s in sequences]

    def _encode(self, seq, aa_to_idx, max_len):
        # Kodowanie aminokwasów na indeksy z paddingiem
        encoded = [aa_to_idx.get(aa, 0) for aa in str(seq)[:max_len]]
        padding = [0] * (max_len - len(encoded))
        return torch.tensor(encoded + padding, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.features[idx], self.labels[idx]

# Słownik mapowania aminokwasów (standardowy)
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i+1 for i, aa in enumerate(amino_acids)}

In [ ]:
# --- PRZYGOTOWANIE EMBEDDINGÓW DLA ZBIORU TESTOWEGO (TYLKO RAZ) ---
print("Generowanie embeddingów ProtBERT dla zbioru testowego...")
X_test_embed = get_embeddings(df_ref.loc[X_test.index, "sequence"].tolist())
X_test_scaled_features = StandardScaler().fit_transform(X_test)
X_test_combined_bert = np.concatenate([X_test_embed, X_test_scaled_features], axis=1)

# Loader dla ProtBERT (Test)
test_ds_bert = AmyloidDataset(df_ref.loc[X_test.index, "sequence"], X_test_combined_bert, y_test, aa_to_idx)
test_loader_bert_final = DataLoader(test_ds_bert, batch_size=64, shuffle=False)

# Standardowy loader dla MLP/CNN (Test) - to już masz
test_ds_standard = AmyloidDataset(df_ref.loc[X_test.index, "sequence"], X_test_scaled_features, y_test, aa_to_idx)
test_loader_standard = DataLoader(test_ds_standard, batch_size=64, shuffle=False)

In [ ]:
results_dl = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# --- GŁÓWNA PĘTLA ---
for data_name, (X_train_curr, y_train_curr) in train_datasets.items():
    print(f"\n" + "="*30)
    print(f"START EKSPERYMENTU DL: {data_name}")
    print("="*30)

    # 1. Przygotowanie sekwencji
    if data_name == "Real Only":
        current_seqs = df_ref.loc[X_train_curr.index, "sequence"]
    elif "ProtGPT2" in data_name:
        current_seqs = pd.concat([df_ref.loc[X_train_ref.index, "sequence"], df_protgpt2["sequence"]])
    elif "cVAE" in data_name:
        current_seqs = pd.concat([df_ref.loc[X_train_ref.index, "sequence"], df_cvae["sequence"]])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_curr)

    # Podział na Train/Val
    idx_tr, idx_val = train_test_split(np.arange(len(y_train_curr)), test_size=0.15, stratify=y_train_curr, random_state=42)

    # 4. ITERACJA PO ARCHITEKTURACH
    deep_models = {
        "MLP": MLP(input_dim=len(selected_features)),
        "CNN-Hybrid": CNNHybrid(vocab_size=len(amino_acids)+1, feat_dim=len(selected_features)),
        "ProtBERT-MLP": ProtBERT_MLP(input_dim=1024 + len(selected_features))
    }

    for m_name, model_obj in deep_models.items():
        print(f"--> Trenowanie {m_name}...")

        if m_name == "ProtBERT-MLP":
            # Przygotowanie danych pod BERT
            X_embed_curr = get_embeddings(current_seqs.tolist())
            X_combined_train = np.concatenate([X_embed_curr, X_train_scaled], axis=1)

            tr_ds = AmyloidDataset(current_seqs.iloc[idx_tr], X_combined_train[idx_tr], y_train_curr.iloc[idx_tr], aa_to_idx)
            vl_ds = AmyloidDataset(current_seqs.iloc[idx_val], X_combined_train[idx_val], y_train_curr.iloc[idx_val], aa_to_idx)

            curr_train_loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
            curr_val_loader = DataLoader(vl_ds, batch_size=64, shuffle=False)
            curr_test_loader = test_loader_bert_final # Używamy BERT-owego testu
        else:
            # Standardowe dane dla MLP/CNN
            tr_ds = AmyloidDataset(current_seqs.iloc[idx_tr], X_train_scaled[idx_tr], y_train_curr.iloc[idx_tr], aa_to_idx)
            vl_ds = AmyloidDataset(current_seqs.iloc[idx_val], X_train_scaled[idx_val], y_train_curr.iloc[idx_val], aa_to_idx)

            curr_train_loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
            curr_val_loader = DataLoader(vl_ds, batch_size=64, shuffle=False)
            curr_test_loader = test_loader_standard # Używamy standardowego testu

        # Trening
        trained_model = run_deep_experiment(model_obj, curr_train_loader, curr_val_loader, device)

        # 5. EWALUACJA (Teraz poza blokiem else, dla każdego modelu)
        trained_model.eval()
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for s_t, f_t, l_t in curr_test_loader:
                s_t, f_t = s_t.to(device), f_t.to(device)

                if m_name in ["MLP", "ProtBERT-MLP"]:
                    logits = trained_model(f_t)
                else:
                    logits = trained_model(s_t, f_t)

                probs = torch.sigmoid(logits).cpu().numpy().flatten()
                all_probs.extend(probs)
                all_labels.extend(l_t.numpy())

        all_probs = np.array(all_probs)
        all_preds = (all_probs > 0.5).astype(int)

        results_dl.append({
            "Dataset": data_name,
            "Model": m_name,
            "AUC": roc_auc_score(all_labels, all_probs),
            "F1": f1_score(all_labels, all_preds),
            "ACC": accuracy_score(all_labels, all_preds)
        })